# SBERT for Quora Question Pairs

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%pip install -q lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 17.7 MB/s eta 0:00:00


In [ ]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torchmetrics.functional.classification import (
    binary_accuracy,
    binary_precision,
    binary_recall,
    binary_f1_score,
)
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
from datasets import load_dataset
import lightning as L
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

In [ ]:
DATASET_NAME = "AlekseyKorshuk/quora-question-pairs"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LOG_DIR = "drive/MyDrive/lms/nlp/assignment"
SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 2
D_MODEL = 384
LEARNING_RATE = 3e-5
MAX_EPOCHS = 3
PATIENCE = 10

## 1. Prepare dataset

In [ ]:
ds = load_dataset(DATASET_NAME, split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dataset_infos.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/404290 [00:00<?, ? examples/s]

In [ ]:
ds

Dataset({
    features: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate'],
    num_rows: 404290
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
ds_cleaned = ds.filter(
    lambda row: isinstance(row["question1"], str) and isinstance(row["question2"], str)
)


def tokenize_row(row):
    output_1 = tokenizer(row["question1"], truncation=True)
    output_2 = tokenizer(row["question2"], truncation=True)
    return {
        "input_ids_1": output_1["input_ids"],
        "attention_mask_1": output_1["attention_mask"],
        "input_ids_2": output_2["input_ids"],
        "attention_mask_2": output_2["attention_mask"],
    }


ds_tokenized = ds_cleaned.map(
    lambda row: tokenize_row(row),
    batched=True,
    remove_columns=["qid1", "qid2", "question1", "question2"],
)

Filter:   0%|          | 0/404290 [00:00<?, ? examples/s]

Map:   0%|          | 0/404287 [00:00<?, ? examples/s]

In [ ]:
ds_split_1 = ds_tokenized.train_test_split(0.3)
ds_split_2 = ds_split_1["test"].train_test_split(0.5)
ds_train, ds_val, ds_test = ds_split_1["train"], ds_split_2["train"], ds_split_2["test"]

In [ ]:
ds_train, ds_val, ds_test

(Dataset({
     features: ['id', 'is_duplicate', 'input_ids_1', 'attention_mask_1', 'input_ids_2', 'attention_mask_2'],
     num_rows: 283000
 }),
 Dataset({
     features: ['id', 'is_duplicate', 'input_ids_1', 'attention_mask_1', 'input_ids_2', 'attention_mask_2'],
     num_rows: 60643
 }),
 Dataset({
     features: ['id', 'is_duplicate', 'input_ids_1', 'attention_mask_1', 'input_ids_2', 'attention_mask_2'],
     num_rows: 60644
 }))

In [ ]:
pad_sequence(
    [torch.zeros(1, 2), torch.zeros(4, 2)],
    batch_first=True,
    padding_value=tokenizer.pad_token_type_id,
).shape

torch.Size([2, 4, 2])

In [ ]:
def data_collator(batch):
    fields = ["input_ids_1", "attention_mask_1", "input_ids_2", "attention_mask_2"]
    data_dict = {}

    for field in ("input_ids_1", "input_ids_2"):
        field_data = [torch.tensor(sample[field]) for sample in batch]
        data_dict[field] = pad_sequence(
            field_data,
            batch_first=True,
            padding_value=tokenizer.pad_token_type_id,
        )

    for field in ("attention_mask_1", "attention_mask_2"):
        field_data = [torch.tensor(sample[field]) for sample in batch]
        data_dict[field] = pad_sequence(
            field_data,
            batch_first=True,
            padding_value=0,
        )

    data_dict["is_duplicate"] = torch.tensor([sample["is_duplicate"] for sample in batch])
    return data_dict

In [ ]:
train_loader = DataLoader(
    ds_train,
    BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)
val_loader = DataLoader(
    ds_val,
    BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)
test_loader = DataLoader(
    ds_test,
    BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)

## 2. Prepare model

In [ ]:
class SbertQuoraModel(L.LightningModule):
    def __init__(self, sbert, d_model=D_MODEL):
        super().__init__()
        self.sbert = sbert
        self.d_model = d_model
        self.classifier = nn.Linear(3 * d_model, 1)

    def mean_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1)
        return (last_hidden_state * mask).sum(1) / mask.sum(1)

    def forward(self, input_ids_1, attention_mask_1, input_ids_2, attention_mask_2):
        output_1 = self.sbert(input_ids_1, attention_mask_1)
        output_2 = self.sbert(input_ids_2, attention_mask_2)

        embed_1 = self.mean_pooling(output_1.last_hidden_state, attention_mask_1)
        embed_2 = self.mean_pooling(output_2.last_hidden_state, attention_mask_2)
        embed = torch.concat((embed_1, embed_2, torch.abs(embed_1 - embed_2)), dim=1)

        logits = self.classifier(embed).reshape(-1)
        return logits

    def training_step(self, batch, batch_idx):
        input_ids_1 = batch["input_ids_1"]
        attention_mask_1 = batch["attention_mask_1"]
        input_ids_2 = batch["input_ids_2"]
        attention_mask_2 = batch["attention_mask_2"]
        labels = batch["is_duplicate"].float()

        logits = self(input_ids_1, attention_mask_1, input_ids_2, attention_mask_2)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        preds = logits > 0

        accuracy = binary_accuracy(preds, labels)
        precision = binary_precision(preds, labels)
        recall = binary_recall(preds, labels)
        f1 = binary_f1_score(preds, labels)

        self.log("train_loss", loss)
        self.log("train_accuracy", accuracy)
        self.log("train_precision", precision)
        self.log("train_recall", recall)
        self.log("train_f1", f1)

        return loss

    def validation_step(self, batch, batch_idx):
        input_ids_1 = batch["input_ids_1"]
        attention_mask_1 = batch["attention_mask_1"]
        input_ids_2 = batch["input_ids_2"]
        attention_mask_2 = batch["attention_mask_2"]
        labels = batch["is_duplicate"].float()

        logits = self(input_ids_1, attention_mask_1, input_ids_2, attention_mask_2)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        preds = logits > 0

        accuracy = binary_accuracy(preds, labels)
        precision = binary_precision(preds, labels)
        recall = binary_recall(preds, labels)
        f1 = binary_f1_score(preds, labels)

        self.log("val_loss", loss)
        self.log("val_accuracy", accuracy)
        self.log("val_precision", precision)
        self.log("val_recall", recall)
        self.log("val_f1", f1)

    def test_step(self, batch, batch_idx):
        input_ids_1 = batch["input_ids_1"]
        attention_mask_1 = batch["attention_mask_1"]
        input_ids_2 = batch["input_ids_2"]
        attention_mask_2 = batch["attention_mask_2"]
        labels = batch["is_duplicate"].float()

        logits = self(input_ids_1, attention_mask_1, input_ids_2, attention_mask_2)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        preds = logits > 0

        accuracy = binary_accuracy(preds, labels)
        precision = binary_precision(preds, labels)
        recall = binary_recall(preds, labels)
        f1 = binary_f1_score(preds, labels)

        self.log("test_loss", loss)
        self.log("test_accuracy", accuracy)
        self.log("test_precision", precision)
        self.log("test_recall", recall)
        self.log("test_f1", f1)

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), LEARNING_RATE)
        return optimizer

## 3. Train model

In [ ]:
sbert = AutoModel.from_pretrained(MODEL_NAME)
sbert.train()
model = SbertQuoraModel(sbert)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
trainer = L.Trainer(
    default_root_dir=LOG_DIR,
    enable_checkpointing=False,
    callbacks=[EarlyStopping("val_loss", patience=PATIENCE)],
    max_epochs=MAX_EPOCHS,
    val_check_interval=500,
    limit_val_batches=500
)
trainer.fit(model, train_loader, val_loader)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ sbert      │ BertModel │ 22.7 M │ train │     0 │
│ 1 │ classifier │ Linear    │  1.2 K │ train │     0 │
└───┴────────────┴───────────┴────────┴───────┴───────┘

Trainable params: 22.7 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 22.7 M                                                                                               
Total estimated model params size (MB): 90.857                                                                     
Modules in train mode: 121                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: `Trainer.fit` stopped: `max_epochs=3` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=3` reached.
